In [1]:
import requests
from bs4 import BeautifulSoup
import csv

In [2]:
url = "https://editorial.rottentomatoes.com/guide/movies-100-percent-score-rotten-tomatoes/"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/153.0.0.0 Safari/537.36"}

In [3]:
response = requests.get(url, headers=headers)
print(response.status_code)

200


In [4]:
soup = BeautifulSoup(response.text, 'html.parser')

In [5]:
movie_card = soup.find_all("div", class_="block-countdown rkv-block")

# Create a csv file
with open('movies.csv', 'w', newline='', encoding = 'utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(["Movie Title", "Release Year", "Tomatometer Score", "Directed By", "Starring", "Critics Consensus", "Synopsis"])

    for movie in movie_card:
        movie_title = movie.find('div', class_="meta-data-wrapper").find('a', class_="meta-title")

        # Movie Title
        if movie_title:
            movie_title = movie_title.get_text(strip=True)
        else:
            movie_title = 'N/A'

        # Release year
        release_year = movie.find('span', class_="meta-year")
        
        if release_year:
            release_year = release_year.get_text(strip=True)
        else:
            release_year = 'N/A'

        # Tomatometer Score
        tomatometer_score = movie.find('div', class_="meta-scores-wrapper").find("span", class_="tMeterScore")
        
        if tomatometer_score:
            tomatometer_score = tomatometer_score.get_text(strip=True)
        else:
            tomatometer_score = 'N/A'
            

        main_class = movie.find('div', class_="meta-details-wrapper")

        p_class = main_class.find_all('p', class_="meta-detail")   # list
        for i in p_class:    
            if p_class:

                # Movie's Director/Directors
                directors = p_class[3].find_all('a')
                directed_by = ", ".join(director.get_text(strip=True) for director in directors)


                # movie's cast
                starring = p_class[2].find_all('a')
                star_cast = ", ".join(cast.get_text(strip=True) for cast in starring)


                # Critics Consensus
                critics_consensus = p_class[0].get_text(strip=True)
                critics_consensus = critics_consensus.replace('Critics Consensus:', "", 1).strip()


                # Synopsis
                movie_link = p_class[1].find('a', class_="full-synopsis")
                if movie_link:
                    movie_url = movie_link['href']

                    response = requests.get(movie_url, headers=headers)
                    soup_movie = BeautifulSoup(response.text, 'html.parser')

                    synopsis = soup_movie.find('rt-text', {'data-qa': 'synopsis-value'})
                    if synopsis:
                        synopsis = synopsis.get_text(strip=True)
                    else:
                        synopsis = 'N/A'

             
            else:
                directed_by = 'N/A'


        writer.writerow([movie_title, release_year, tomatometer_score, directed_by, star_cast, critics_consensus, synopsis])